# XMCA-Net-Lite: 05 - Retrieval Consistency Scoring (S_R)
### Purpose
Retrieve passages, rerank using MiniLM cross-encoder, and compute Retrieval Consistency Score (S_R).
Implements both:
- Learned isotonic retrieval fusion (standard)
- Fixed weighted fusion (for ablation study)


In [ ]:
import sys
from pathlib import Path
import numpy as np
# Robust project root discovery (supports Kaggle uppercase/lowercase and local)
PROJECT_ROOT = None
for candidate in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/XMCA"), Path("/kaggle/working/xmca")]:
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate.resolve()
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, set_seed
from src.retrieval.indexer import DenseRetriever
from src.retrieval.reranker import MiniLMReranker
from src.retrieval.fusion import RetrievalScoreFusion
from src.utils.checkpoints import ShardManager
from src.utils.logging_metadata import save_run_metadata

config = load_config()
seed = config["project"]["default_seed"]
set_seed(seed)


In [ ]:
corpora_dir = Path(config["paths"]["corpora_dir"])
retriever = DenseRetriever()
retriever.load(corpora_dir / "faiss_index.bin", corpora_dir / "corpus_metadata.json")
reranker = MiniLMReranker(model_id=config["models"]["reranker"]["model_id"])
fusion_isotonic = RetrievalScoreFusion(mode="isotonic")
fusion_fixed = RetrievalScoreFusion(mode="fixed_weighted")

qwen_gen_dir = Path(config["paths"]["generations_dir"]) / "qwen2vl"
gen_df = ShardManager(qwen_gen_dir).load_all_shards("qwen2vl_generation")
out_dir = Path(config["paths"]["retrieval_dir"])
shard_mgr_sr = ShardManager(out_dir, shard_size=config["execution"]["shard_size"])

items = gen_df.to_dict(orient="records")
remaining = shard_mgr_sr.get_remaining_items(items, id_key="id")
total_rem = len(remaining)
print(f"Items to compute S_R: {total_rem}", flush=True)

shard_records = []
shard_idx = 0

for i, row in enumerate(remaining, 1):
    search_results = retriever.search(row["question"], top_k=config["models"]["retriever"]["top_k"])
    passages = [doc["text"] for doc, _ in search_results]
    dense_scores = [score for _, score in search_results]
    
    rerank_scores = reranker.score_pairs(row["question"], passages)
    s_r_isotonic = fusion_isotonic.fuse(dense_scores, rerank_scores)
    s_r_fixed = fusion_fixed.fuse(dense_scores, rerank_scores)
    
    shard_records.append({
        "id": row["id"],
        "dataset": row["dataset"],
        "retrieved_passages": passages,
        "dense_scores": dense_scores,
        "rerank_scores": rerank_scores,
        "s_r": s_r_isotonic,
        "s_r_fixed": s_r_fixed,
        "is_hallucination_gt": row["is_hallucination_gt"]
    })
    
    if i % 50 == 0 or i == total_rem:
        pct = (i / total_rem) * 100
        print(f"Retrieval scoring (S_R): {i}/{total_rem} done ({pct:.1f}%) [Shard {shard_idx}]", flush=True)

    if len(shard_records) >= config["execution"]["shard_size"]:
        shard_mgr_sr.save_shard(shard_idx, shard_records, prefix="retrieval_scores")
        print(f"✓ Saved retrieval shard {shard_idx:04d} ({len(shard_records)} items saved, {i}/{total_rem} done).", flush=True)
        shard_records = []
        shard_idx += 1

if shard_records:
    shard_mgr_sr.save_shard(shard_idx, shard_records, prefix="retrieval_scores")
    print(f"✓ Saved final retrieval shard {shard_idx:04d} ({total_rem}/{total_rem} done).", flush=True)

save_run_metadata(out_dir, config, seed, "05_retrieval", {"total_processed": len(items)})
print("\nStep 05 completed. Retrieval scores saved.", flush=True)